# Optional · TMVA demo — selecting events the way physicists say it
**Data:** MAGIC gamma telescope (`data/magic/magic04.data`): 19 020 simulated events, ten image parameters, class `g` (gamma, signal) or `h` (hadron, background).
**Goal:** train a BDT with TMVA, then translate machine-learning words into physics words, and optimise the cut with the **Asimov significance**.

| Machine learning | Physics |
|---|---|
| recall, true-positive rate | **signal efficiency** $\\varepsilon_s$ |
| false-positive rate | **background efficiency** $\\varepsilon_b$ (rejection $=1-\\varepsilon_b$) |
| precision | **purity** $\\pi=s/(s+b)$ |
| accuracy | not used: classes are unbalanced and priors are not the real ones |

Efficiencies do **not** depend on the signal-to-background ratio. Purity does.

In [ ]:
import ROOT
import numpy as np
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name
import pandas as pd

## 1 · Data and the ROOT files TMVA needs

In [ ]:
VARS = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym", "fM3Long", "fM3Trans", "fAlpha", "fDist"]
df = pd.read_csv("../data/magic/magic04.data", names=VARS + ["class"])
print(df["class"].value_counts())

for cls, name in (("g", "sig"), ("h", "bkg")):
    part = df[df["class"] == cls]
    rdf = ROOT.RDF.FromNumpy({v: part[v].to_numpy(dtype=np.float32) for v in VARS})
    rdf.Snapshot(name, "magic_%s.root" % name)

## 2 · Train a BDT (and a rectangular-cuts baseline)

In [ ]:
out = ROOT.TFile.Open("TMVA_magic.root", "RECREATE")
factory = ROOT.TMVA.Factory("TMVAMagic", out, "!V:!Silent:Color:DrawProgressBar:AnalysisType=Classification")
loader = ROOT.TMVA.DataLoader("dataset_magic")
for v in VARS:
    loader.AddVariable(v, "F")

fs, fb = ROOT.TFile.Open("magic_sig.root"), ROOT.TFile.Open("magic_bkg.root")
loader.AddSignalTree(fs.Get("sig"), 1.0)
loader.AddBackgroundTree(fb.Get("bkg"), 1.0)
loader.PrepareTrainingAndTestTree(ROOT.TCut(""), "SplitMode=Random:NormMode=NumEvents:!V")

factory.BookMethod(loader, ROOT.TMVA.Types.kBDT, "BDT",
                   "!V:NTrees=300:MaxDepth=3:BoostType=AdaBoost:AdaBoostBeta=0.5:SeparationType=GiniIndex:nCuts=20")
factory.BookMethod(loader, ROOT.TMVA.Types.kFisher, "Fisher", "!V:Fisher")
factory.TrainAllMethods()
factory.TestAllMethods()
factory.EvaluateAllMethods()
out.Close()

TMVA writes its standard plots (ROC curve, response distributions, variable correlations) in `TMVA_magic.root`
and in `dataset_magic/`. Open the ROC curve and the response distributions with the **TMVA GUI** (`ROOT.TMVA.TMVAGui("TMVA_magic.root")`) or read the test tree below.

## 3 · The physicist's analysis of the response

In [ ]:
rdf = ROOT.RDataFrame("dataset_magic/TestTree", "TMVA_magic.root")
arr = rdf.AsNumpy(["classID", "BDT"])
cls, score = arr["classID"], arr["BDT"]          # classID 0 = signal, 1 = background
is_s, is_b = cls == 0, cls == 1
print("test events: %d signal, %d background" % (is_s.sum(), is_b.sum()))

In [ ]:
def scan(score, is_s, is_b, S_tot, B_tot, ncut=200):
    """For each cut (score > c): efficiencies, rejection, purity and two significances."""
    cuts = np.linspace(score.min(), score.max(), ncut)
    eff_s = np.array([np.mean(score[is_s] > c) for c in cuts])
    eff_b = np.array([np.mean(score[is_b] > c) for c in cuts])
    s, b = S_tot * eff_s, B_tot * eff_b
    with np.errstate(divide="ignore", invalid="ignore"):
        purity = s / (s + b)
        z_simple = s / np.sqrt(b)
        z_asimov = np.sqrt(2 * ((s + b) * np.log(1 + s / b) - s))
    return cuts, eff_s, eff_b, purity, z_simple, z_asimov

# ILLUSTRATIVE yields before any cut: 50 signal events and 500 background events
cuts, eff_s, eff_b, purity, z_simple, z_asimov = scan(score, is_s, is_b, S_tot=50.0, B_tot=500.0)
i_simple, i_asimov = np.nanargmax(np.where(np.isfinite(z_simple), z_simple, -1)), np.nanargmax(np.where(np.isfinite(z_asimov), z_asimov, -1))
print("cut maximising s/sqrt(b): %.2f  (eff_s %.2f, eff_b %.3f)" % (cuts[i_simple], eff_s[i_simple], eff_b[i_simple]))
print("cut maximising Z_A      : %.2f  (eff_s %.2f, eff_b %.3f)" % (cuts[i_asimov], eff_s[i_asimov], eff_b[i_asimov]))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(eff_s, 1 - eff_b); ax[0].set_xlabel("signal efficiency"); ax[0].set_ylabel("background rejection"); ax[0].set_title("ROC")
ax[1].plot(cuts, eff_s, label="signal eff"); ax[1].plot(cuts, 1 - eff_b, label="bkg rejection"); ax[1].plot(cuts, purity, label="purity (S=50, B=500)")
ax[1].set_xlabel("BDT cut"); ax[1].legend()
ax[2].plot(cuts, z_simple, label="s/sqrt(b)"); ax[2].plot(cuts, z_asimov, label="Z_A (Asimov)"); ax[2].set_xlabel("BDT cut"); ax[2].legend()
ax[2].set_ylim(0, np.nanmax(z_asimov[np.isfinite(z_asimov)]) * 1.2)
plt.tight_layout(); plt.show()

In [ ]:
ok = np.isfinite(z_asimov) & np.isfinite(z_simple)
check("signal efficiency falls with the cut", eff_s[0] >= eff_s[-1])
check("Z_A never exceeds s/sqrt(b): the naive formula always overestimates", np.all(z_asimov[ok] <= z_simple[ok] + 1e-9))
check("the optimal cut leaves a useful signal efficiency", 0.05 < eff_s[i_asimov] < 0.99)

## Questions
1. Why does `s/sqrt(b)` push the cut to extreme values when $\\varepsilon_b\\to0$, and what does $Z_A$ do instead?
2. Change the illustrative yields: $(S,B)=(50,50000)$, then $(500,500)$. How do the optimal cut and the two significances move?
3. Your cut was optimised on the test sample, then used on data. What bias does that introduce, and how do you avoid it?
4. The efficiencies depend on the MC. Which systematic uncertainty would you add to the model, and as which kind of nuisance parameter?

**Reference:** Cowan, Cranmer, Gross, Vitells, *Asymptotic formulae for likelihood-based tests of new physics*, Eur. Phys. J. C71 (2011) 1554, arXiv:1007.1727.